# 1. Introduction

`dsfb-bank` is the executable empirical companion for the DSFB theorem banks. This notebook rebuilds the crate from scratch, runs the full theorem-bank demo suite from scratch, captures the fresh timestamped output directory from the current session, validates the artifact inventory, and generates empirical figures directly from those fresh CSV outputs.

The notebook deliberately avoids cached results. All tables and figures are derived from the current session's run only.

# 2. Environment Setup

This section locates the repository root, installs required Python packages if the Colab session does not already have them, and ensures the notebook is operating relative to the DSFB workspace root.

In [ ]:
import datetime as dt
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "pandas", "matplotlib"], check=True)
    import pandas as pd
    import matplotlib.pyplot as plt

from IPython.display import display, Markdown

REPO_URL = "https://github.com/infinityabundance/dsfb.git"
SESSION_UTC = dt.datetime.utcnow().isoformat() + "Z"


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        cargo = candidate / "Cargo.toml"
        if cargo.exists() and "[workspace]" in cargo.read_text():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    clone_root = Path("/content/dsfb")
    if not clone_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_root)], check=True)
    repo_root = clone_root

os.chdir(repo_root)
display(Markdown(f"Repository root: `{repo_root}`"))
display(Markdown(f"Notebook session UTC start: `{SESSION_UTC}`"))

# 3. Fresh Rust Installation

This section installs Rust in the current session if needed, then verifies `cargo` and `rustc` before building `dsfb-bank`.

In [ ]:
if shutil.which("cargo") is None or shutil.which("rustc") is None:
    subprocess.run("curl https://sh.rustup.rs -sSf | sh -s -- -y", shell=True, check=True)
    cargo_bin = Path.home() / ".cargo" / "bin"
    os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"

subprocess.run(["cargo", "--version"], check=True, cwd=repo_root)
subprocess.run(["rustc", "--version"], check=True, cwd=repo_root)

# 4. Fresh Build of dsfb-bank

The crate is cleaned and rebuilt from scratch in the current session. The clean step is explicit and isolated to the `dsfb-bank` package.

In [ ]:
subprocess.run(["cargo", "clean", "-p", "dsfb-bank"], check=True, cwd=repo_root)
subprocess.run(["cargo", "build", "-p", "dsfb-bank"], check=True, cwd=repo_root)

# 5. Fresh Theorem Bank Run

This section executes the full theorem-bank run from scratch and captures the fresh timestamped output directory printed by the crate.

In [ ]:
run_started_utc = dt.datetime.utcnow().isoformat() + "Z"
run_proc = subprocess.run(
    ["cargo", "run", "-p", "dsfb-bank", "--", "--all"],
    cwd=repo_root,
    text=True,
    capture_output=True,
    check=True,
)
print(run_proc.stdout)
if run_proc.stderr.strip():
    print(run_proc.stderr)

stdout_lines = [line.strip() for line in run_proc.stdout.splitlines() if line.strip()]
assert stdout_lines, "dsfb-bank did not print an output directory"
run_dir = Path(stdout_lines[-1]).resolve()
assert run_dir.exists(), f"Fresh run directory does not exist: {run_dir}"
display(Markdown(f"Fresh run directory from current execution: `{run_dir}`"))

# 6. Locate Fresh Output Directory

The notebook verifies that the newest timestamped directory under `output-dsfb-bank/` is the one created by the current run, and it uses only that directory for the rest of the analysis.

In [ ]:
output_root = repo_root / "output-dsfb-bank"
all_run_dirs = sorted([path for path in output_root.iterdir() if path.is_dir()])
assert all_run_dirs, "No timestamped output directories were found"
newest_run_dir = max(all_run_dirs, key=lambda path: path.name)
assert newest_run_dir == run_dir, f"Newest run dir {newest_run_dir} does not match current run dir {run_dir}"
display(Markdown(f"Newest timestamped run directory: `{newest_run_dir}`"))

# 7. Validate Output Inventory

This section validates the fresh artifact inventory, including the manifest, the run summary, per-component theorem CSV counts, and the realization-space outputs required by the crate contract.

In [ ]:
manifest_path = run_dir / "manifest.json"
run_summary_path = run_dir / "run_summary.md"
logs_path = run_dir / "logs.txt"

assert manifest_path.exists(), f"Missing manifest: {manifest_path}"
assert run_summary_path.exists(), f"Missing run summary: {run_summary_path}"
assert logs_path.exists(), f"Missing logs: {logs_path}"

required_counts = {
    "core": 11,
    "dsfb": 20,
    "dscd": 20,
    "tmtr": 20,
    "add": 20,
    "srd": 20,
    "hret": 20,
}
component_csv_files = {}
for component, expected in required_counts.items():
    component_dir = run_dir / component
    assert component_dir.exists(), f"Missing component directory: {component_dir}"
    csv_files = sorted(component_dir.glob("*.csv"))
    component_csv_files[component] = csv_files
    assert len(csv_files) == expected, f"Expected {expected} CSVs in {component}, found {len(csv_files)}"

realizations_dir = run_dir / "realizations"
required_realization_files = [
    realizations_dir / "dsfb_realizations.csv",
    realizations_dir / "dscd_realizations.csv",
    realizations_dir / "tmtr_realizations.csv",
    realizations_dir / "add_realizations.csv",
    realizations_dir / "srd_realizations.csv",
    realizations_dir / "hret_realizations.csv",
    realizations_dir / "all_realizations.csv",
]
for path in required_realization_files:
    assert path.exists(), f"Missing realization CSV: {path}"

manifest = json.loads(manifest_path.read_text())
assert len(manifest["theorem_demos_run"]) == 131, "Expected 131 theorem demos in manifest"
display(Markdown("Fresh output inventory validation passed."))

# 8. Load CSV Outputs

All CSVs are loaded dynamically from the fresh run directory created above. No cached outputs are used.

In [ ]:
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.2

component_frames = {}
for component, paths in component_csv_files.items():
    frames = {}
    for path in paths:
        df = pd.read_csv(path)
        df["source_csv"] = str(path.relative_to(run_dir))
        frames[path.name] = df
    component_frames[component] = frames

realization_frames = {}
for path in required_realization_files:
    df = pd.read_csv(path)
    df["source_csv"] = str(path.relative_to(run_dir))
    realization_frames[path.name] = df

summary_rows = []
for component, frames in component_frames.items():
    for name, df in frames.items():
        summary_rows.append(
            {
                "component": component,
                "file_name": name,
                "source_csv": str((run_dir / component / name).relative_to(run_dir)),
                "theorem_id": df["theorem_id"].iloc[0],
                "theorem_name": df["theorem_name"].iloc[0],
                "case_count": len(df),
                "pass_count": int(df["pass"].sum()),
                "pass_rate": float(df["pass"].mean()),
            }
        )

theorem_index = pd.DataFrame(summary_rows)
assert theorem_index.shape[0] == 131, f"Expected 131 theorem CSVs, found {theorem_index.shape[0]}"


def get_csv(component: str, prefix: str) -> tuple[Path, pd.DataFrame]:
    for name, df in component_frames[component].items():
        if name.startswith(prefix):
            return run_dir / component / name, df.copy()
    raise KeyError(f"Missing {component} CSV with prefix {prefix}")

figure_manifest = []
figure_output_dir = run_dir / "figure_pngs"
figure_output_dir.mkdir(parents=True, exist_ok=True)
saved_figure_pngs = []
display(Markdown(f"Figure PNG output directory for this run: `{figure_output_dir}`"))


def figure_slug(title: str) -> str:
    slug = "".join(ch.lower() if ch.isalnum() else "_" for ch in title)
    while "__" in slug:
        slug = slug.replace("__", "_")
    return slug.strip("_")


def save_registered_figure(fig):
    assert figure_manifest, "register_figure must be called before saving a figure"
    meta = figure_manifest[-1]
    file_name = f"figure_{meta['figure_number']:02d}_{figure_slug(meta['title'])}.png"
    output_path = figure_output_dir / file_name
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    meta["png_path"] = str(output_path.relative_to(run_dir))
    if output_path not in saved_figure_pngs:
        saved_figure_pngs.append(output_path)
    return output_path


def register_figure(number: int, title: str, source_paths, theorem_families: str, explanation: str):
    rel_paths = [str(Path(path).relative_to(run_dir)) for path in source_paths]
    display(Markdown(
        f"### Figure {number} — {title}\n"
        f"Source CSV path(s): `{'`, `'.join(rel_paths)}`\n\n"
        f"{explanation}"
    ))
    figure_manifest.append(
        {
            "figure_number": number,
            "title": title,
            "source_csvs": ", ".join(rel_paths),
            "theorem_families_covered": theorem_families,
        }
    )


def category_codes(series: pd.Series):
    categories = list(pd.Index(series.astype(str)).unique())
    mapping = {value: idx for idx, value in enumerate(categories)}
    return series.astype(str).map(mapping), mapping

# 9. Generate Theorem Bank Coverage Figures

These figures verify theorem-bank coverage, theorem counts, and pass summaries directly from the fresh run outputs.

In [ ]:
components_order = ["dsfb", "dscd", "tmtr", "add", "srd", "hret"]
coverage = theorem_index[theorem_index["component"].isin(components_order)].copy()
pivot = coverage.pivot(index="theorem_id", columns="component", values="case_count").fillna(0)
pivot = pivot[[component for component in components_order if component in pivot.columns]]
register_figure(
    1,
    "Theorem Bank Coverage Heatmap",
    [run_dir / row for row in coverage["source_csv"].tolist()],
    "DSFB, DSCD, TMTR, ADD, SRD, HRET",
    "This heatmap shows that every theorem-bank theorem emitted empirical witness cases in the current run."
)
fig, ax = plt.subplots(figsize=(8, 14))
im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
ax.set_xlabel("Component")
ax.set_ylabel("Theorem ID")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=6)
ax.set_title("Figure 1 — Theorem Bank Coverage Heatmap\nSources: fresh theorem-bank CSVs from current run")
fig.colorbar(im, ax=ax, label="Number of cases")
save_registered_figure(fig)
plt.show()

count_df = coverage.groupby("component").size().reindex(components_order)
register_figure(
    2,
    "Theorem Count by Component",
    [run_dir / row for row in coverage["source_csv"].tolist()],
    "DSFB, DSCD, TMTR, ADD, SRD, HRET",
    "This bar chart confirms the expected 20-theorem coverage for each theorem bank in the current run."
)
fig, ax = plt.subplots()
count_df.plot(kind="bar", ax=ax, color=["#2d6a4f", "#40916c", "#52796f", "#b56576", "#6d597a", "#3a86ff"])
ax.set_xlabel("Component")
ax.set_ylabel("Theorem count")
ax.set_title("Figure 2 — Theorem Count by Component\nSources: fresh theorem-bank CSV inventory")
save_registered_figure(fig)
plt.show()

pass_summary = coverage.groupby("component")["pass_rate"].mean().reindex(components_order)
register_figure(
    3,
    "Theorem Pass/Fail Summary",
    [run_dir / row for row in coverage["source_csv"].tolist()],
    "DSFB, DSCD, TMTR, ADD, SRD, HRET",
    "This figure reports empirical pass rates from the current theorem-bank witness runs without using cached data."
)
fig, ax = plt.subplots()
pass_summary.plot(kind="bar", ax=ax, color="#bc6c25")
ax.set_xlabel("Component")
ax.set_ylabel("Average pass rate")
ax.set_ylim(0.0, 1.05)
ax.set_title("Figure 3 — Theorem Pass/Fail Summary\nSources: fresh theorem-bank CSV inventory")
save_registered_figure(fig)
plt.show()

# 10. Generate Behavior Figures

These figures visualize the deterministic witness behavior for DSFB, DSCD, TMTR, ADD, SRD, and HRET using only CSVs from the fresh run directory.

In [ ]:
# Figure 4
path_dsfb_02, dsfb_02 = get_csv("dsfb", "02_")
recoverability = dsfb_02.groupby("case_type")["equivalence_class_size"].mean().reindex(["injective", "non_injective"])
register_figure(
    4,
    "DSFB Recoverability vs Ambiguity",
    [path_dsfb_02],
    "DSFB injectivity and observability theorems",
    "This figure contrasts singleton inverse images under injective forward maps with ambiguous observation classes under non-injective contrast cases."
)
fig, ax = plt.subplots()
recoverability.plot(kind="bar", ax=ax, color=["#1b4332", "#d00000"])
ax.set_xlabel("Case type")
ax.set_ylabel("Average observation-class size")
ax.set_title(f"Figure 4 — DSFB Recoverability vs Ambiguity\nSource: {path_dsfb_02.relative_to(run_dir)}")
save_registered_figure(fig)
plt.show()

# Figure 5
path_dsfb_07, dsfb_07 = get_csv("dsfb", "07_")
register_figure(
    5,
    "DSFB Residual Collapse",
    [path_dsfb_07],
    "DSFB residual theorems",
    "This figure shows residual collapse on exact-image cases and the nonzero contrast case outside exact reconstruction semantics."
)
fig, ax = plt.subplots()
ax.plot(range(len(dsfb_07)), dsfb_07["residual_value"], marker="o")
ax.set_xticks(range(len(dsfb_07)))
ax.set_xticklabels(dsfb_07["case_id"], rotation=45, ha="right")
ax.set_xlabel("Case ID")
ax.set_ylabel("Residual value")
ax.set_title(f"Figure 5 — DSFB Residual Collapse\nSource: {path_dsfb_07.relative_to(run_dir)}")
plt.tight_layout()
save_registered_figure(fig)
plt.show()

# Figure 6
path_dsfb_20, dsfb_20 = get_csv("dsfb", "20_")
dsfb_20 = dsfb_20.sort_values("time_step")
dsfb_20["reconstructed_state_numeric"] = dsfb_20["reconstructed_state_id"].str.extract(r"(\d+)").astype(float)
register_figure(
    6,
    "DSFB Periodic Observation / Reconstruction",
    [path_dsfb_20],
    "DSFB periodicity theorems",
    "This time series shows periodic exact-image observations and the matching periodic reconstructed structure from the current run."
)
fig, ax = plt.subplots()
ax.plot(dsfb_20["time_step"], dsfb_20["observation_value"], marker="o", label="observation_value")
ax.plot(dsfb_20["time_step"], dsfb_20["reconstructed_state_numeric"], marker="s", label="reconstructed_state")
ax.set_xlabel("Time step")
ax.set_ylabel("Observation / reconstructed state")
ax.set_title(f"Figure 6 — DSFB Periodic Observation / Reconstruction\nSource: {path_dsfb_20.relative_to(run_dir)}")
ax.legend()
save_registered_figure(fig)
plt.show()

# Figure 7
path_dscd_08, dscd_08 = get_csv("dscd", "08_")
dscd_scatter = dscd_08[["graph_id", "node_count", "longest_path"]].drop_duplicates()
register_figure(
    7,
    "DSCD DAG Size vs Longest Path",
    [path_dscd_08],
    "DSCD path-bound theorems",
    "This scatter plot empirically illustrates the finite DAG path-length bound against the line y = node_count - 1."
)
fig, ax = plt.subplots()
ax.scatter(dscd_scatter["node_count"], dscd_scatter["longest_path"], color="#005f73", s=60)
xs = sorted(dscd_scatter["node_count"].unique())
ax.plot(xs, [x - 1 for x in xs], color="#ae2012", linestyle="--", label="y = node_count - 1")
ax.set_xlabel("Node count")
ax.set_ylabel("Longest path")
ax.set_title(f"Figure 7 — DSCD DAG Size vs Longest Path\nSource: {path_dscd_08.relative_to(run_dir)}")
ax.legend()
save_registered_figure(fig)
plt.show()

# Figure 8
path_dscd_13, dscd_13 = get_csv("dscd", "13_")
dscd_reduction = dscd_13[["graph_id", "edge_count", "reduction_edge_count", "reachability_count"]].drop_duplicates()
register_figure(
    8,
    "DSCD Reduction vs Reachability Preservation",
    [path_dscd_13],
    "DSCD transitive-reduction theorems",
    "This comparison shows edge-count reduction without reachability loss in the current transitive-reduction witnesses."
)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(dscd_reduction["graph_id"], dscd_reduction["edge_count"], label="original_edge_count", color="#468faf")
axes[0].bar(dscd_reduction["graph_id"], dscd_reduction["reduction_edge_count"], label="reduced_edge_count", alpha=0.8, color="#014f86")
axes[0].set_xlabel("Graph ID")
axes[0].set_ylabel("Edge count")
axes[0].set_title("Original vs reduced edges")
axes[0].legend()
axes[1].bar(dscd_reduction["graph_id"], dscd_reduction["reachability_count"], color="#ffb703")
axes[1].set_xlabel("Graph ID")
axes[1].set_ylabel("Reachability count")
axes[1].set_title("Reachability preserved")
fig.suptitle(f"Figure 8 — DSCD Reduction vs Reachability Preservation\nSource: {path_dscd_13.relative_to(run_dir)}")
plt.tight_layout()
save_registered_figure(fig)
plt.show()

# Figure 9
path_tmtr_01, tmtr_01 = get_csv("tmtr", "01_")
path_tmtr_08, tmtr_08 = get_csv("tmtr", "08_")
tmtr_traces = pd.concat([tmtr_01, tmtr_08], ignore_index=True)
register_figure(
    9,
    "TMTR Trust Descent Trajectories",
    [path_tmtr_01, path_tmtr_08],
    "TMTR monotonicity and periodicity theorems",
    "This figure plots multiple TMTR witness orbits to show monotone descent and trust-neutral cyclic behavior."
)
fig, ax = plt.subplots()
for orbit_id, frame in tmtr_traces.groupby("orbit_id"):
    frame = frame.sort_values("iteration")
    ax.plot(frame["iteration"], frame["trust_value"], marker="o", label=orbit_id)
ax.set_xlabel("Iteration")
ax.set_ylabel("Trust value")
ax.set_title(
    "Figure 9 — TMTR Trust Descent Trajectories\n"
    f"Sources: {path_tmtr_01.relative_to(run_dir)}, {path_tmtr_08.relative_to(run_dir)}"
)
ax.legend()
save_registered_figure(fig)
plt.show()

# Figure 10
stabilization = theorem_index[theorem_index["component"] == "tmtr"].copy()
stabilization_rows = []
for name, df in component_frames["tmtr"].items():
    stabilization_rows.extend(df[["orbit_id", "stabilization_iteration"]].drop_duplicates().to_dict("records"))
stabilization_df = pd.DataFrame(stabilization_rows).drop_duplicates()
register_figure(
    10,
    "TMTR Stabilization Iteration Distribution",
    [run_dir / row for row in theorem_index[theorem_index["component"] == "tmtr"]["source_csv"].tolist()],
    "TMTR stabilization theorems",
    "This histogram summarizes how quickly the deterministic TMTR witness orbits stabilize in the current run."
)
fig, ax = plt.subplots()
ax.hist(stabilization_df["stabilization_iteration"], bins=range(int(stabilization_df["stabilization_iteration"].min()), int(stabilization_df["stabilization_iteration"].max()) + 2), color="#6a994e", align="left")
ax.set_xlabel("Stabilization iteration")
ax.set_ylabel("Orbit count")
ax.set_title("Figure 10 — TMTR Stabilization Iteration Distribution\nSources: fresh TMTR theorem CSVs")
save_registered_figure(fig)
plt.show()

# Figure 11
path_add_02, add_02 = get_csv("add", "02_")
register_figure(
    11,
    "ADD Residual Threshold Detection",
    [path_add_02],
    "ADD residual-threshold theorems",
    "This time series shows the deterministic residual threshold detector firing on fresh signal data from the current run."
)
fig, ax = plt.subplots()
ax.plot(add_02["time_step"], add_02["signal_value"], marker="o", label="signal_value")
ax.plot(add_02["time_step"], add_02["residual_value"], marker="s", label="residual_value")
ax.plot(add_02["time_step"], add_02["threshold"], linestyle="--", label="threshold")
trigger_mask = add_02["detector_output"].str.contains("residual")
ax.scatter(add_02.loc[trigger_mask, "time_step"], add_02.loc[trigger_mask, "residual_value"], color="#d00000", label="detector_output")
ax.set_xlabel("Time step")
ax.set_ylabel("Signal / residual / threshold")
ax.set_title(f"Figure 11 — ADD Residual Threshold Detection\nSource: {path_add_02.relative_to(run_dir)}")
ax.legend()
save_registered_figure(fig)
plt.show()

# Figure 12
path_add_03, add_03 = get_csv("add", "03_")
path_add_09, add_09 = get_csv("add", "09_")
path_add_10, add_10 = get_csv("add", "10_")
register_figure(
    12,
    "ADD Difference and Curvature Detection",
    [path_add_03, path_add_09, path_add_10],
    "ADD drift, step, and spike theorems",
    "This figure compares deterministic drift, step, and spike witnesses through signal, first-difference, and second-difference views."
)
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
axes[0].plot(add_03["time_step"], add_03["signal_value"], marker="o", label="drift signal")
axes[0].plot(add_09["time_step"], add_09["signal_value"], marker="s", label="step signal")
axes[0].plot(add_10["time_step"], add_10["signal_value"], marker="^", label="spike signal")
axes[0].set_ylabel("Signal value")
axes[0].legend()
axes[1].plot(add_03["time_step"], add_03["first_difference"], marker="o", label="drift first difference")
axes[1].plot(add_09["time_step"], add_09["first_difference"], marker="s", label="step first difference")
axes[1].set_ylabel("First difference")
axes[1].legend()
axes[2].plot(add_10["time_step"], add_10["second_difference"], marker="^", label="spike second difference")
axes[2].set_xlabel("Time step")
axes[2].set_ylabel("Second difference")
axes[2].legend()
fig.suptitle(
    "Figure 12 — ADD Difference and Curvature Detection\n"
    f"Sources: {path_add_03.relative_to(run_dir)}, {path_add_09.relative_to(run_dir)}, {path_add_10.relative_to(run_dir)}"
)
plt.tight_layout()
save_registered_figure(fig)
plt.show()

# Figure 13
path_srd_03, srd_03 = get_csv("srd", "03_")
srd_timeline = srd_03[srd_03["trajectory_id"] == "block_constant"].copy()
fine_codes, fine_map = category_codes(srd_timeline["fine_regime"])
register_figure(
    13,
    "SRD Regime Transition Timeline",
    [path_srd_03],
    "SRD transition-detection theorems",
    "This timeline visualizes fine regime segmentation and marks the transition events detected in the current run."
)
fig, ax = plt.subplots()
ax.step(srd_timeline["time_step"], fine_codes, where="post", label="fine_regime")
transition_points = srd_timeline[srd_timeline["transition_flag"]]
ax.scatter(transition_points["time_step"], fine_codes.loc[transition_points.index], color="#d00000", label="transition_flag")
ax.set_xlabel("Time step")
ax.set_ylabel("Fine regime code")
ax.set_yticks(list(fine_map.values()))
ax.set_yticklabels(list(fine_map.keys()))
ax.set_title(f"Figure 13 — SRD Regime Transition Timeline\nSource: {path_srd_03.relative_to(run_dir)}")
ax.legend()
save_registered_figure(fig)
plt.show()

# Figure 14
path_srd_18, srd_18 = get_csv("srd", "18_")
srd_coarse = srd_18[srd_18["trajectory_id"] == "block_constant"].copy()
coarse_codes, coarse_map = category_codes(srd_coarse["coarse_regime"])
register_figure(
    14,
    "SRD Coarsened Regime Timeline",
    [path_srd_18],
    "SRD coarsening theorems",
    "This timeline shows the coarsened regime labels derived from the fresh run and highlights deterministic coarsening behavior."
)
fig, ax = plt.subplots()
ax.step(srd_coarse["time_step"], coarse_codes, where="post")
ax.set_xlabel("Time step")
ax.set_ylabel("Coarse regime code")
ax.set_yticks(list(coarse_map.values()))
ax.set_yticklabels(list(coarse_map.keys()))
ax.set_title(f"Figure 14 — SRD Coarsened Regime Timeline\nSource: {path_srd_18.relative_to(run_dir)}")
save_registered_figure(fig)
plt.show()

# Figure 15
path_hret_10, hret_10 = get_csv("hret", "10_")
hret_success = hret_10.groupby(["trace_id", "trace_length"], as_index=False)["reconstruction_success"].mean()
register_figure(
    15,
    "HRET Trace Reconstruction Success",
    [path_hret_10],
    "HRET reconstruction theorems",
    "This figure summarizes exact historical reconstruction success as a function of trace length using only the current run's HRET outputs."
)
fig, ax = plt.subplots()
ax.scatter(hret_success["trace_length"], hret_success["reconstruction_success"], color="#7b2cbf", s=70)
ax.set_xlabel("Trace length")
ax.set_ylabel("Reconstruction success rate")
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"Figure 15 — HRET Trace Reconstruction Success\nSource: {path_hret_10.relative_to(run_dir)}")
save_registered_figure(fig)
plt.show()

# 11. Generate Cross-Stack Figure

These figures visualize the end-to-end DSFB pipeline trace and the core-theorem empirical dashboard from the fresh run outputs.

In [ ]:
# Figure 16
path_core_10, core_10 = get_csv("core", "grand_unification")
register_figure(
    16,
    "Full DSFB Pipeline Trace",
    [path_core_10],
    "Core theorem layer and cross-stack DSFB pipeline",
    "This figure shows the shared time-axis behavior of signal, reconstructed state, trust, regime, and anomaly outputs for the integrated DSFB stack."
)
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
axes[0].plot(core_10["time_step"], core_10["signal_value"], marker="o")
axes[0].set_ylabel("Signal value")
axes[1].plot(core_10["time_step"], core_10["reconstructed_state"], marker="s")
axes[1].set_ylabel("Reconstructed state")
axes[2].plot(core_10["time_step"], core_10["trust_value"], marker="^")
axes[2].set_ylabel("Trust value")
regime_codes, regime_map = category_codes(core_10["regime_label"])
axes[3].step(core_10["time_step"], regime_codes, where="post", label="regime_label")
axes[3].scatter(core_10.loc[core_10["anomaly_flag"], "time_step"], regime_codes.loc[core_10["anomaly_flag"]], color="#d00000", label="anomaly_flag")
axes[3].set_ylabel("Regime code")
axes[3].set_xlabel("Time step")
axes[3].set_yticks(list(regime_map.values()))
axes[3].set_yticklabels(list(regime_map.keys()))
axes[3].legend()
fig.suptitle(f"Figure 16 — Full DSFB Pipeline Trace\nSource: {path_core_10.relative_to(run_dir)}")
plt.tight_layout()
save_registered_figure(fig)
plt.show()

# Figure 17
core_index = theorem_index[theorem_index["component"] == "core"].copy()
core_heat = core_index.set_index("theorem_id")[["pass_rate"]]
register_figure(
    17,
    "Core Theorem Dashboard",
    [run_dir / row for row in core_index["source_csv"].tolist()],
    "Core theorem layer",
    "This dashboard summarizes the empirical status of the core theorem layer from the fresh run's core CSV outputs."
)
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(core_heat.values, aspect="auto", cmap="magma", vmin=0.0, vmax=1.0)
ax.set_xlabel("Pass rate")
ax.set_ylabel("Core theorem ID")
ax.set_xticks([0])
ax.set_xticklabels(["pass_rate"])
ax.set_yticks(range(len(core_heat.index)))
ax.set_yticklabels(core_heat.index)
ax.set_title("Figure 17 — Core Theorem Dashboard\nSources: fresh core theorem CSVs")
fig.colorbar(im, ax=ax, label="Pass rate")
save_registered_figure(fig)
plt.show()

# 12. Display Summary Tables

The notebook now displays the fresh CSV inventory, theorem counts, pass summaries, and the figure manifest generated in this session.

In [ ]:
csv_inventory = pd.DataFrame(
    [
        {"relative_path": str(path.relative_to(run_dir)), "bytes": path.stat().st_size}
        for path in sorted(run_dir.rglob("*.csv"))
    ]
)

theorem_counts = theorem_index.groupby("component").size().reset_index(name="theorem_count")
pass_counts = theorem_index.groupby("component").agg(
    theorem_csvs=("theorem_id", "count"),
    total_rows=("case_count", "sum"),
    pass_rows=("pass_count", "sum"),
).reset_index()
pass_counts["fail_rows"] = pass_counts["total_rows"] - pass_counts["pass_rows"]
figure_manifest_df = pd.DataFrame(figure_manifest).sort_values("figure_number")
png_inventory = pd.DataFrame(
    [
        {"relative_path": str(path.relative_to(run_dir)), "bytes": path.stat().st_size}
        for path in sorted(figure_output_dir.glob("*.png"))
    ]
)

display(Markdown("#### CSV inventory table"))
display(csv_inventory)

display(Markdown(f"#### Figure PNG output folder\n`{figure_output_dir}`"))
display(png_inventory)

display(Markdown("#### Theorem counts by component"))
display(theorem_counts)

display(Markdown("#### Pass/fail counts by component"))
display(pass_counts)

display(Markdown("#### Figure manifest table"))
display(figure_manifest_df)

# 13. Reproducibility Confirmation

The final confirmation below is derived from the current session variables and the fresh run directory captured above.

In [ ]:
display(Markdown(
    f"The crate was built in the current notebook session.\n\n"
    f"The theorem bank was executed in the current session.\n\n"
    f"The output directory used was created during this session: `{run_dir}`.\n\n"
    f"All figures and tables above were generated from those fresh outputs only.\n\n"
    f"PNG figure exports were also written during this session to: `{figure_output_dir}`."
))